# Research08: Generative vs Traditional Time-Series Augmentation

This notebook compares generative anomaly augmentation methods with traditional transformation-based time-series augmentation methods under the same downstream anomaly-detection setting.

In [1]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score, f1_score, fbeta_score, precision_score, recall_score, roc_auc_score

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'tools'))

from research05_balanced_aug_experiment import (
    comprehensive_augmentation,
    ensure_window_3d,
    flatten_windows,
    frequency_domain,
    load_npz_x,
    load_npz_xy,
    magnitude_warp,
    noise_injection,
    sample_rows,
    time_warp,
)

RANDOM_STATE = 42
NORMAL_TRAIN_SIZE = 5000
AUGMENTATION_COUNT = 1000
THRESHOLDS = np.linspace(0.05, 0.95, 91)
FBETA_BETA = 2.0

RESULT_DIR = ROOT / 'data' / 'research08' / 'results'
FIGURE_DIR = ROOT / 'data' / 'research08' / 'figures'
RESULT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style='whitegrid')

In [2]:
split_dir = ROOT / 'data' / 'augmentation_split'
generated_dir = ROOT / 'data' / 'research02' / 'generated'

x_normal_train, _ = load_npz_xy(split_dir / 'normal_train_windows.npz')
x_anomaly_seed, _ = load_npz_xy(split_dir / 'anomaly_train_seed_windows.npz')
x_val, y_val = load_npz_xy(split_dir / 'validation_windows.npz')
x_final, y_final = load_npz_xy(split_dir / 'final_test_windows.npz')

split_summary = pd.DataFrame([
    {'split': 'normal_train', 'windows': len(x_normal_train), 'normal': len(x_normal_train), 'anomaly': 0},
    {'split': 'anomaly_train_seed', 'windows': len(x_anomaly_seed), 'normal': 0, 'anomaly': len(x_anomaly_seed)},
    {'split': 'validation', 'windows': len(x_val), 'normal': int((y_val == 0).sum()), 'anomaly': int((y_val == 1).sum())},
    {'split': 'final_test', 'windows': len(x_final), 'normal': int((y_final == 0).sum()), 'anomaly': int((y_final == 1).sum())},
])
split_summary

,split,windows,normal,anomaly
0,normal_train,135730,135730,0
1,anomaly_train_seed,247,0,247
2,validation,1447,1283,164
3,final_test,3081,2785,296


In [3]:
def select_threshold_by_validation_f1(y_true, anomaly_proba):
    best = None
    for threshold in THRESHOLDS:
        pred = (anomaly_proba >= threshold).astype(np.int8)
        candidate = {
            'threshold': float(threshold),
            'validation_precision': precision_score(y_true, pred, zero_division=0),
            'validation_recall': recall_score(y_true, pred, zero_division=0),
            'validation_f1': f1_score(y_true, pred, zero_division=0),
            'validation_f2': fbeta_score(y_true, pred, beta=FBETA_BETA, zero_division=0),
        }
        if best is None or (candidate['validation_f1'], candidate['validation_recall'], candidate['validation_precision']) > (best['validation_f1'], best['validation_recall'], best['validation_precision']):
            best = candidate
    return best

def evaluate_augmented_method(method, augmentation_family, x_augmented, seed):
    x_normal_sample = sample_rows(x_normal_train, NORMAL_TRAIN_SIZE, seed=seed + 23)
    x_augmented = sample_rows(ensure_window_3d(x_augmented), AUGMENTATION_COUNT, seed=seed + 11)
    x_anomaly_train = np.concatenate([x_anomaly_seed, x_augmented], axis=0)
    x_train = np.concatenate([x_normal_sample, x_anomaly_train], axis=0)
    y_train = np.concatenate([
        np.zeros(len(x_normal_sample), dtype=np.int8),
        np.ones(len(x_anomaly_train), dtype=np.int8),
    ])

    model = RandomForestClassifier(
        n_estimators=300,
        min_samples_leaf=2,
        class_weight='balanced_subsample',
        random_state=seed,
        n_jobs=-1,
    )
    model.fit(flatten_windows(x_train), y_train)

    val_proba = model.predict_proba(flatten_windows(x_val))[:, 1]
    threshold_result = select_threshold_by_validation_f1(y_val, val_proba)
    threshold = threshold_result['threshold']

    final_proba = model.predict_proba(flatten_windows(x_final))[:, 1]
    final_pred = (final_proba >= threshold).astype(np.int8)
    return {
        'method': method,
        'augmentation_family': augmentation_family,
        'augmentation_count': AUGMENTATION_COUNT,
        'normal_train_used': int(len(x_normal_sample)),
        'real_anomaly_seed_used': int(len(x_anomaly_seed)),
        'augmented_anomaly_used': int(len(x_augmented)),
        'total_anomaly_train_used': int(len(x_anomaly_train)),
        'normal_to_anomaly_ratio': float(len(x_normal_sample) / len(x_anomaly_train)),
        'threshold': threshold,
        **{k: float(v) for k, v in threshold_result.items() if k != 'threshold'},
        'precision': precision_score(y_final, final_pred, zero_division=0),
        'recall': recall_score(y_final, final_pred, zero_division=0),
        'f1': f1_score(y_final, final_pred, zero_division=0),
        'f2': fbeta_score(y_final, final_pred, beta=FBETA_BETA, zero_division=0),
        'auroc': roc_auc_score(y_final, final_proba),
        'auprc': average_precision_score(y_final, final_proba),
        'false_negative': int(((y_final == 1) & (final_pred == 0)).sum()),
        'false_positive': int(((y_final == 0) & (final_pred == 1)).sum()),
    }

In [4]:
augmentation_sets = {
    'GT-GAN': ('generative', load_npz_x(generated_dir / 'gtgan_series_windows.npz')),
    'Diffusion': ('generative', load_npz_x(generated_dir / 'diffusion_series_windows.npz')),
    'Masking GT-GAN': ('generative', load_npz_x(generated_dir / 'gtgan_masked_windows.npz')),
    'Masking Diffusion': ('generative', load_npz_x(generated_dir / 'diffusion_masked_windows.npz')),
    'Magnitude warping': ('traditional', magnitude_warp(x_anomaly_seed, AUGMENTATION_COUNT, RANDOM_STATE + 102)),
    'Time warping': ('traditional', time_warp(x_anomaly_seed, AUGMENTATION_COUNT, RANDOM_STATE + 101)),
    'Noise injection': ('traditional', noise_injection(x_anomaly_seed, AUGMENTATION_COUNT, RANDOM_STATE + 103)),
    'Frequency domain': ('traditional', frequency_domain(x_anomaly_seed, AUGMENTATION_COUNT, RANDOM_STATE + 104)),
    'Comprehensive': ('traditional', comprehensive_augmentation(x_anomaly_seed, AUGMENTATION_COUNT, RANDOM_STATE + 105)),
}

rows = []
for idx, (method, (family, x_augmented)) in enumerate(augmentation_sets.items()):
    rows.append(evaluate_augmented_method(method, family, x_augmented, RANDOM_STATE + 100 * (idx + 1)))

results_df = pd.DataFrame(rows).sort_values('f1', ascending=False).reset_index(drop=True)
results_df.to_csv(RESULT_DIR / 'research08_generative_vs_traditional.csv', index=False)
results_df

,method,augmentation_family,augmentation_count,normal_train_used,real_anomaly_seed_used,augmented_anomaly_used,total_anomaly_train_used,normal_to_anomaly_ratio,threshold,validation_precision,...,validation_f1,validation_f2,precision,recall,f1,f2,auroc,auprc,false_negative,false_positive
0,Noise injection,traditional,1000,5000,247,1000,1247,4.009623,0.83,1.000000,...,0.828571,0.751295,0.991803,0.817568,0.896296,0.847339,0.995939,0.969403,54,2
1,Magnitude warping,traditional,1000,5000,247,1000,1247,4.009623,0.81,0.865248,...,0.800000,0.765370,0.930403,0.858108,0.892794,0.871654,0.995184,0.965001,42,19
2,Comprehensive,traditional,1000,5000,247,1000,1247,4.009623,0.85,0.926230,...,0.790210,0.726221,0.969828,0.760135,0.852273,0.794492,0.994253,0.956376,71,7
3,Frequency domain,traditional,1000,5000,247,1000,1247,4.009623,0.82,0.804196,...,0.749186,0.719650,0.829932,0.824324,0.827119,0.825440,0.990616,0.919686,52,50
4,Masking Diffusion,generative,1000,5000,247,1000,1247,4.009623,0.89,0.850394,...,0.742268,0.689655,0.855932,0.682432,0.759398,0.711268,0.983100,0.887787,94,34
5,Masking GT-GAN,generative,1000,5000,247,1000,1247,4.009623,0.91,0.887931,...,0.735714,0.667098,0.890411,0.658784,0.757282,0.694939,0.968913,0.853761,101,24
6,GT-GAN,generative,1000,5000,247,1000,1247,4.009623,0.87,0.885246,...,0.755245,0.694087,0.894231,0.628378,0.738095,0.668103,0.980084,0.864029,110,22
7,Diffusion,generative,1000,5000,247,1000,1247,4.009623,0.87,0.923077,...,0.768683,0.698577,0.904040,0.604730,0.724696,0.647612,0.981706,0.877154,117,19
8,Time warping,traditional,1000,5000,247,1000,1247,4.009623,0.86,0.744828,...,0.699029,0.674157,0.730627,0.668919,0.698413,0.680412,0.960914,0.798171,98,73


In [5]:
family_summary = results_df.groupby('augmentation_family').agg(
    method_count=('method', 'count'),
    best_method=('method', lambda s: results_df.loc[s.index, :].sort_values('f1', ascending=False).iloc[0]['method']),
    precision_mean=('precision', 'mean'),
    recall_mean=('recall', 'mean'),
    f1_mean=('f1', 'mean'),
    f2_mean=('f2', 'mean'),
    auroc_mean=('auroc', 'mean'),
    auprc_mean=('auprc', 'mean'),
    f1_max=('f1', 'max'),
    f2_max=('f2', 'max'),
    auprc_max=('auprc', 'max'),
).reset_index()
family_summary.to_csv(RESULT_DIR / 'research08_family_summary.csv', index=False)
family_summary

,augmentation_family,method_count,best_method,precision_mean,recall_mean,f1_mean,f2_mean,auroc_mean,auprc_mean,f1_max,f2_max,auprc_max
0,generative,4,Masking Diffusion,0.886154,0.643581,0.744868,0.680481,0.978451,0.870683,0.759398,0.711268,0.887787
1,traditional,5,Noise injection,0.890519,0.785811,0.833379,0.803867,0.987381,0.921727,0.896296,0.871654,0.969403


In [6]:
plot_df = results_df.melt(
    id_vars=['method', 'augmentation_family'],
    value_vars=['precision', 'recall', 'f1', 'f2', 'auprc'],
    var_name='metric',
    value_name='score',
)
plt.figure(figsize=(12, 6))
sns.barplot(data=plot_df, x='method', y='score', hue='metric')
plt.xticks(rotation=35, ha='right')
plt.ylim(0, 1.05)
plt.xlabel('')
plt.ylabel('Final test score')
plt.title('Generative vs Traditional Time-Series Augmentation')
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'research08_generative_vs_traditional_metrics.png', dpi=200)
plt.show()

C:\Users\gram\AppData\Local\Temp\ipykernel_18036\3987626826.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [7]:
summary = {
    'setting': 'Imbalanced training condition with 5000 normal windows, 247 real anomaly seed windows, and 1000 augmented anomaly windows.',
    'model': 'RandomForestClassifier(n_estimators=300, min_samples_leaf=2, class_weight=balanced_subsample)',
    'threshold_selection': 'Threshold selected by maximum F1 on the real validation split.',
    'final_test': split_summary.to_dict(orient='records')[-1],
    'best_method_by_f1': results_df.iloc[0].to_dict(),
    'best_generative_by_f1': results_df[results_df['augmentation_family'] == 'generative'].iloc[0].to_dict(),
    'best_traditional_by_f1': results_df[results_df['augmentation_family'] == 'traditional'].iloc[0].to_dict(),
    'family_summary': family_summary.to_dict(orient='records'),
}
with open(RESULT_DIR / 'research08_summary.json', 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)
summary

{'setting': 'Imbalanced training condition with 5000 normal windows, 247 real anomaly seed windows, and 1000 augmented anomaly windows.',
 'model': 'RandomForestClassifier(n_estimators=300, min_samples_leaf=2, class_weight=balanced_subsample)',
 'threshold_selection': 'Threshold selected by maximum F1 on the real validation split.',
 'final_test': {'split': 'final_test',
  'windows': 3081,
  'normal': 2785,
  'anomaly': 296},
 'best_method_by_f1': {'method': 'Noise injection',
  'augmentation_family': 'traditional',
  'augmentation_count': 1000,
  'normal_train_used': 5000,
  'real_anomaly_seed_used': 247,
  'augmented_anomaly_used': 1000,
  'total_anomaly_train_used': 1247,
  'normal_to_anomaly_ratio': 4.0096230954290295,
  'threshold': 0.83,
  'validation_precision': 1.0,
  'validation_recall': 0.7073170731707317,
  'validation_f1': 0.8285714285714286,
  'validation_f2': 0.7512953367875648,
  'precision': 0.9918032786885246,
  'recall': 0.8175675675675675,
  'f1': 0.8962962962962963,